# RE_202 · Qui acabarà comprant?

**Bloc 2 · Supervisat · repte 2 de 6 · durada: 1 o 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes a l'equip de dades d'una botiga en línia. Màrqueting no vol una caixa negra que
«endevini»: vol un model que **expliqui** quines senyals de la navegació empenyen una sessió cap a
la compra, i que doni una **probabilitat** que es pugui posar en un tauler.

La pregunta: **amb una regressió logística, quines variables pesen a favor i en contra de la compra,
quant, i com arriba el model a la probabilitat d'una sessió concreta?**

L'enunciat complet, amb el diccionari de columnes, és a [RE_202](../RE_202_compres_en_linia.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- La regressió logística, la sigmoide i els pesos: [ML_04 §2-§4](../../01_teoria/ML_04_regressio_logistica.ipynb)
- Un model entrenat és un vector i una multiplicació: [MA_01 §7](../../03_matematiques/MA_01_algebra_lineal.ipynb)
- La log-loss, la funció que minimitza: [MA_02 §5](../../03_matematiques/MA_02_descens_gradient.ipynb)
- `StandardScaler` i el parany de `fit` contra `transform`: [EX_00 §2](../../02_practica/EX_00_caixa_eines.ipynb)
- `OneHotEncoder`, `ColumnTransformer` i `DummyClassifier`: [EX_00 §4, §6 i §7](../../02_practica/EX_00_caixa_eines.ipynb)

## Les dades

**Online Shoppers Purchasing Intention** (Sakar i Kastro, 2018; UCI), a OpenML amb `data_id=46947`.
Llicència CC BY 4.0. Cada fila és **una sessió** d'un visitant en una botiga en línia durant un any:
quantes pàgines de cada tipus ha vist i quant de temps, mètriques de Google Analytics (`BounceRates`,
`ExitRates`, `PageValues`), el mes, el tipus de visitant... La columna `Revenue` diu si la sessió ha
acabat en compra. No hi ha cap identificador de persona.

Documentació de cada columna: <https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=46947, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Forma, tipus i objectiu

Forma, tipus de cada columna i proporció de sessions amb compra. Converteix l'objectiu en un `y` de 0 i 1 i comprova que no hi has perdut res pel camí.

In [ ]:
print(df.dtypes)
print(df["Revenue"].value_counts())
y = (df["Revenue"].astype(str) == "True").astype(int)
assert len(y) == 12330 and set(y.unique()) == {0, 1}
assert y.sum() == (df["Revenue"].astype(str) == "True").sum()
print("Proporció de compres:", round(y.mean(), 4))

> **Resultat.** 1.908 compres de 12.330 sessions (15,5 %). `Revenue` és categòrica amb valors `True`/`False` escrits com a text: si es compara amb el booleà `True` sense convertir-la, peta o surt tot 0.

### 1.2 · Duplicats i mesos

Quantes files duplicades hi ha? Són errors o poden ser sessions diferents que s'assemblen? Mira la columna `Month`: quins valors té, quants mesos hi apareixen i com estan escrits?

In [ ]:
print("Duplicats:", df.duplicated().sum())
print(df[df.duplicated(keep=False)][["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]].describe().round(3))
print(df["Month"].value_counts())
assert df["Month"].value_counts().sum() == len(df)

> **Resultat.** 125 duplicats: són sessions mínimes (les 201 files implicades tenen una o dues pàgines, durada 0, `ExitRates` 0,2 i `PageValues` 0), que és normal que coincideixin. Esborrar-los no és obvi. `Month` té 10 valors: falten gener i abril, i `June` va escrit sencer quan la resta són tres lletres. Que noti l'absència de dos mesos és el que importa (el model no sabrà què fer amb un gener).

### 1.3 · Números que no són números

`OperatingSystems`, `Browser`, `Region` i `TrafficType` tenen valors numèrics. Ho són de veritat? Decideix quines columnes entraran com a numèriques i quines com a categòriques, i escriu-ho en dues llistes `num` i `cat`.

In [ ]:
for c in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    print(c, df[c].nunique(), list(df[c].value_counts().index[:6]))
cat = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
num = [c for c in df.columns if c not in cat + ["Revenue"]]
assert not set(num) & set(cat) and len(num) + len(cat) + 1 == df.shape[1]
print("num:", num)

> **Resultat.** Són codis: el 4 no és més gran que el 2. Van a `cat` amb `Month`, `VisitorType` i `Weekend`. Queden 10 numèriques.

## Nivell 2 · Respondre

### 2.1 · La línia de sortida

Parteix en entrenament i test (25 %, estratificat, `random_state=42`). Quina precisió té un model que sempre diu «no compra»? Guarda-la: tot el que facis l'ha de superar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X = df.drop(columns="Revenue")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train[num], y_train)
pred_d = dummy.predict(X_test[num])
assert len(np.unique(pred_d)) == 1
acc_dummy = dummy.score(X_test[num], y_test)
assert np.isclose(acc_dummy, 1 - y_test.mean())
print("Precisió del model que no diu mai compra:", round(acc_dummy, 4))

> **Resultat.** El model ximple fa 0,845. Qualsevol precisió per sota d'això és pitjor que no fer res.

### 2.2 · Regressió logística amb les numèriques

Escala les columnes `num` (ajustant l'escalador **només amb l'entrenament**) i entrena una regressió logística. Quina precisió treu a test? I comparada amb la línia de sortida?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

sc = StandardScaler().fit(X_train[num])
Xtr, Xte = sc.transform(X_train[num]), sc.transform(X_test[num])
assert np.allclose(Xtr.mean(axis=0), 0, atol=1e-6) and np.allclose(Xtr.std(axis=0), 1, atol=1e-6)
print("mitjanes del test escalat:", Xte.mean(axis=0).round(3))

lr = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
print("Precisió test:", round(lr.score(Xte, y_test), 4), "| línia de sortida:", round(acc_dummy, 4))

> **Resultat.** La logística arriba a 0,884 de test (contra 0,845), unes quatre centèsimes per sobre del model ximple. Sembla poc: la discussió sobre si la precisió és la mesura bona aquí és la del RE_203.

### 2.3 · La teoria, al codi: la probabilitat a mà

Sense fer servir `predict_proba`, calcula la probabilitat de compra de **totes** les sessions de test amb els pesos del model: primer $z = X\,w + b$, després la sigmoide. Compara-ho amb `predict_proba`. Després, tria una sessió de test i escriu, columna a columna, quant aporta cada variable a la seva $z$.

In [ ]:
w, b = lr.coef_.ravel(), lr.intercept_[0]
z = Xte @ w + b
p_ma = 1 / (1 + np.exp(-z))
assert np.allclose(p_ma, lr.predict_proba(Xte)[:, 1])
assert ((p_ma > 0) & (p_ma < 1)).all()

i = int(np.argmax(p_ma))                     # la sessió que el model veu més clara
aport = pd.Series(Xte[i] * w, index=num).sort_values()
assert np.isclose(aport.sum() + b, z[i])
print(f"sessió {i}: z = {z[i]:.3f}, p = {p_ma[i]:.3f}, compra real = {y_test.iloc[i]}")
print(aport.round(3))

> **Resultat.** Les probabilitats a mà coincideixen amb `predict_proba` (mateix vector, fins als decimals). En la sessió amb probabilitat més alta, gairebé tota la $z$ ve d'una sola columna, com es veu a la descomposició.

### 2.4 · Què ha après el model

Fes una taula (i un gràfic de barres) dels pesos ordenats. Quines variables empenyen cap a la compra i quines en contra? Per què **només** es poden comparar els pesos entre si perquè les columnes estan escalades? Tradueix el pes més gran a una frase: «si aquesta variable puja una desviació típica, les *odds* de compra es multipliquen per ...».

In [ ]:
pesos = pd.Series(w, index=num).sort_values()
assert len(pesos) == len(num)
assert ((np.exp(pesos) > 1) == (pesos > 0)).all()
print(pd.DataFrame({"pes": pesos, "x odds per 1 desv.": np.exp(pesos)}).round(3))
pesos.plot.barh(); plt.axvline(0, color="k", lw=0.8); plt.title("Pesos (columnes escalades)"); plt.show()

> **Resultat.** `PageValues` té de llarg el pes més gran i positiu (pes 1,54: les *odds* es multipliquen per 4,7 per cada desviació típica); `ExitRates` és el més negatiu (−0,80, *odds* × 0,45). La resta són petits. Aquesta és la troballa del repte.

### 2.5 · La teoria, al codi: la log-loss

Implementa la log-loss (l'entropia creuada binària) amb NumPy, amb les probabilitats de l'2.3, i compara-la amb `log_loss` de scikit-learn. Calcula també la log-loss d'un «model» que dona a totes les sessions la mateixa probabilitat: la proporció de compres de l'entrenament. Què et diu la comparació que la precisió no et deia?

In [ ]:
from sklearn.metrics import log_loss

def la_meva_log_loss(yv, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(yv * np.log(p) + (1 - yv) * np.log(1 - p))

yt = y_test.values
ll = la_meva_log_loss(yt, p_ma)
assert np.isclose(ll, log_loss(yt, p_ma)) and ll > 0
p_const = np.full(len(yt), y_train.mean())
print(f"log-loss model: {ll:.4f} | sklearn: {log_loss(yt, p_ma):.4f} | constant: {la_meva_log_loss(yt, p_const):.4f}")

> **Resultat.** Log-loss del model 0,305 contra 0,431 del model constant: la millora és molt més clara que en precisió, perquè la log-loss premia que les probabilitats estiguin ben posades, no només el costat del 0,5.

## Nivell 3 · Més enllà

### 3.1 · Afegeix les categòriques

Fes un `ColumnTransformer` que escali `num` i codifiqui `cat` amb one-hot, i entrena la mateixa logística dins d'un `Pipeline`. Millora la log-loss? Quins mesos o tipus de visitant pesen més? Què faria el model amb una sessió del gener?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

Xtr_c, Xte_c = X_train.copy(), X_test.copy()
Xtr_c[cat] = Xtr_c[cat].astype(str); Xte_c[cat] = Xte_c[cat].astype(str)
pre = ColumnTransformer([("num", StandardScaler(), num),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
pipe = make_pipeline(pre, LogisticRegression(max_iter=2000)).fit(Xtr_c, y_train)
p2 = pipe.predict_proba(Xte_c)[:, 1]
print("log-loss amb categòriques:", round(log_loss(yt, p2), 4), "| precisió:", round(pipe.score(Xte_c, y_test), 4))
pw = pd.Series(pipe[-1].coef_.ravel(), index=pipe[:-1].get_feature_names_out())
print(pw[pw.index.str.contains("Month|VisitorType")].sort_values().round(3))

> **Resultat.** La log-loss millora poc (de 0,305 a 0,294) i la precisió fins i tot baixa una mica (0,882): bon exemple que les dues mesures no sempre van juntes. Novembre té el pes més alt dels mesos (0,71) i febrer el més baix (−1,20). Un gener faria totes les columnes de mes 0, és a dir, el model el tractaria com «cap mes en concret».

### 3.2 · Sense la variable estrella

Treu la variable amb el pes més gran en valor absolut i torna a entrenar. Quant empitjora? Llegeix a la documentació **com es calcula** aquesta variable: es coneix en el moment en què voldries fer servir el model (per exemple, a mitja sessió, per oferir un descompte)?

In [ ]:
estrella = pesos.abs().idxmax()
num2 = [c for c in num if c != estrella]
sc2 = StandardScaler().fit(X_train[num2])
lr2 = LogisticRegression(max_iter=1000).fit(sc2.transform(X_train[num2]), y_train)
p3 = lr2.predict_proba(sc2.transform(X_test[num2]))[:, 1]
print(f"sense {estrella}: log-loss {log_loss(yt, p3):.4f}, precisió {lr2.score(sc2.transform(X_test[num2]), y_test):.4f}")

> **Resultat.** Sense `PageValues` la log-loss puja a 0,385 i la precisió (0,844) queda **per sota** de la línia de sortida (0,845). `PageValues` és el valor mitjà de les pàgines vistes **abans d'una transacció**: es calcula amb informació de la mateixa sessió i, per tant, en part, de la compra. És el debat de *leakage* que tornarà al Bloc 3.

## Conclusió

La logística supera el model que no diu mai «compra» (0,884 contra 0,845 de precisió), i la
millora es veu molt més clara amb la log-loss (0,305 contra 0,431). La probabilitat calculada a mà,
$z = Xw + b$ i la sigmoide, coincideix amb `predict_proba`. Una sola variable fa gairebé tota la
feina: `PageValues` (pes 1,54, *odds* × 4,7 per desviació típica), i `ExitRates` és la que més empeny
en contra (*odds* × 0,45). Sense `PageValues` el model cau per sota de la línia de sortida, i com
que es calcula amb les pàgines vistes abans d'una transacció, no es coneix a mitja sessió: abans de
posar el model en un tauler, cal decidir si es pot fer servir en el moment d'actuar.